In [3]:
from langchain_community.document_loaders import DirectoryLoader
from sec_html_loader import SECHTMLLoader


/var/folders/zr/kmnfygcd58d30464r3x3mm600000gn/T/ipykernel_59279/2458594646.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader
/Users/kushalraju/Desktop/AI Engineer/projects/FinSight_AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
### Text Loader for directory load
from langchain_community.document_loaders import DirectoryLoader
dir_loader= DirectoryLoader("data/raw",
                            glob="**/*.html",
                            loader_cls=SECHTMLLoader,
                            show_progress=True)
documents=dir_loader.load()
documents

  0%|          | 0/10 [00:00<?, ?it/s]/Users/kushalraju/Desktop/AI Engineer/projects/FinSight_AI/sec_html_loader.py:477: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")
/Users/kushalraju/Desktop/AI Engineer/projects/FinSight_AI/sec_html_loader.py:642: XMLParsedAsHTMLWarning: It looks lik

[Document(metadata={'source': 'data/raw/AMZN/AMZN_10K_2025.html', 'company': 'AMZN', 'filing_type': '10-K', 'filing_year': '2025', 'content_type': 'text'}, page_content='UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n____________________________________\nFORM\n____________________________________\n(Mark One)\nFor the fiscal year ended\nor\nFor the transition period from to .\nCommission File No.\n____________________________________\n(Exact name of registrant as specified in its charter)\n,\n(\n)\n(Address and telephone number, including area code, of registrant’s principal executive offices)\nSecurities registered pursuant to Section 12(b) of the Act:\nSecurities registered pursuant to Section 12(g) of the Act:\nNone\n____________________________________\nIndicate by check mark if the registrant is a well-known seasoned issuer, as defined in Rule 405 of the Securities Act.\n☒\nNo\n☐\nIndicate by check mark if the registrant is not required to file reports p

In [5]:
text_documents= [doc 
for doc in documents
if doc.metadata.get('content_type')== 'text'
]

table_documents= [doc 
for doc in documents
if doc.metadata.get('content_type')== 'table'
]

In [6]:
print(f'Number of text documents :{len(text_documents)}')
print(f'Number of table documents:{len(table_documents)}')

Number of text documents :10
Number of table documents:763


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=100):
    text_splitters = RecursiveCharacterTextSplitter(   
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    
    split_docs = text_splitters.split_documents(documents)
    print(f"Total number of documents before splitting:{len(documents)} and after splitting: {len(split_docs)}")
    return split_docs

In [7]:
text_chunks=split_documents(text_documents)
text_chunks[0:5]

Total number of documents before splitting:10 and after splitting: 1896


[Document(metadata={'source': 'data/raw/AMZN/AMZN_10K_2025.html', 'company': 'AMZN', 'filing_type': '10-K', 'filing_year': '2025', 'content_type': 'text'}, page_content='UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n____________________________________\nFORM\n____________________________________\n(Mark One)\nFor the fiscal year ended\nor\nFor the transition period from to .\nCommission File No.\n____________________________________\n(Exact name of registrant as specified in its charter)\n,\n(\n)\n(Address and telephone number, including area code, of registrant’s principal executive offices)\nSecurities registered pursuant to Section 12(b) of the Act:\nSecurities registered pursuant to Section 12(g) of the Act:\nNone\n____________________________________\nIndicate by check mark if the registrant is a well-known seasoned issuer, as defined in Rule 405 of the Securities Act.\n☒\nNo\n☐\nIndicate by check mark if the registrant is not required to file reports p

In [8]:
chunks = text_chunks + table_documents
print(f"Total no of chunks are : {len(chunks)}")

Total no of chunks are : 2659


In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

class EmbeddingManager:
    """ Handles embedding generation using sentence-transformers. """

    def __init__(self,model_name: str="all-mpnet-base-v2"):
        """ Initializes the embedding manager. 
        Args: model_name : HuggingFace model name for sentence-transformers."""
        
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """ Loads the sentence-transformers model. """
        try:
            print(f"Loading embedding model {self.model_name} in short time")
            self.model= SentenceTransformer(self.model_name)
            print(f"Loaded embedded model {self.model_name} succesfully with embedding dimension {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Failed to load model {self.model_name} {e}")
            raise
    
    def generate_embeddings(self,text: list[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.

        Args: texts: list of texts to embed

        Returns: numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")           
        print(f"Generating embedding for {len(text)} texts")
        embeddings=self.model.encode(text,show_progress_bar= True)
        print(f"Gnerated embeddings with shape : {embeddings.shape}")
        return embeddings
    

embeddingmanager = EmbeddingManager()
embeddingmanager

Loading embedding model all-mpnet-base-v2 in short time


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9532.94it/s]


Loaded embedded model all-mpnet-base-v2 succesfully with embedding dimension 768


In [10]:
### Convert text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate embeddings for the text chunks
embeddings=embeddingmanager.generate_embeddings(texts)


Generating embedding for 2659 texts


Batches: 100%|██████████| 84/84 [01:13<00:00,  1.14it/s]

Gnerated embeddings with shape : (2659, 768)


In [11]:
print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("Embedding dimension:", embeddings.shape[1])

Number of chunks: 2659
Embedding shape: (2659, 768)
Embedding dimension: 768


In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
pinecone_api_key = os.environ.get('PINECONE_API_KEY')


In [13]:
from pinecone import Pinecone,ServerlessSpec
import uuid

pc = Pinecone(api_key=pinecone_api_key)

INDEX_NAME = "finsight"

if INDEX_NAME not in [index["name"] for index in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    print("Created Pinecone index:", INDEX_NAME)

else:
    print("Index already exists:", INDEX_NAME)

index = pc.Index(INDEX_NAME)

print("Connected to index:", INDEX_NAME)


vectors = []

for i, doc in enumerate(chunks):

    vector = {
        "id": f"doc_{uuid.uuid4().hex[:8]}_{i}",
        "values": embeddings[i].tolist(),
        "metadata": {
            "company": doc.metadata.get("company"),
            "filing_type": doc.metadata.get("filing_type"),
            "filing_year": doc.metadata.get("filing_year"),
            "content_type": doc.metadata.get("content_type"),
            "text": doc.page_content
        }
    }
    vectors.append(vector)

Created Pinecone index: finsight
Connected to index: finsight


In [18]:
print("Total vectors:", len(vectors))
print("Vector dimension:", len(vectors[0]["values"]))
print("Metadata:", vectors[0]["metadata"])

NameError: name 'vectors' is not defined

In [15]:
batch_size = 100

for i in range(0, len(vectors), batch_size):

    batch = vectors[i:i + batch_size]

    response = index.upsert(
        vectors=batch
    )

    print(
        f"Uploaded {min(i + batch_size, len(vectors))}/{len(vectors)}"
    )

Uploaded 100/2659
Uploaded 200/2659
Uploaded 300/2659
Uploaded 400/2659
Uploaded 500/2659
Uploaded 600/2659
Uploaded 700/2659
Uploaded 800/2659
Uploaded 900/2659
Uploaded 1000/2659
Uploaded 1100/2659
Uploaded 1200/2659
Uploaded 1300/2659
Uploaded 1400/2659
Uploaded 1500/2659
Uploaded 1600/2659
Uploaded 1700/2659
Uploaded 1800/2659
Uploaded 1900/2659
Uploaded 2000/2659
Uploaded 2100/2659
Uploaded 2200/2659
Uploaded 2300/2659
Uploaded 2400/2659
Uploaded 2500/2659
Uploaded 2600/2659
Uploaded 2659/2659


In [16]:
stats = index.describe_index_stats()

print(stats)

DescribeIndexStatsResponse(dimension=768, total_vector_count=2659, metric='cosine', namespaces=1)


In [16]:
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

NameError: name 'load_dotenv' is not defined

In [18]:
from typing import Literal
from pydantic import BaseModel, Field


class SearchQuery(BaseModel):
    intent: Literal[
        "question_answering",
        "filing_retrieval"
    ] = Field(
        description="The user's intended action."
    )

    company: Literal[
        "AAPL",
        "MSFT",
        "AMZN",
        "NVDA",
        "GOOGL"
    ] = Field(
        description="The company ticker."
    )

    filing_year: str = Field(
        description="The requested SEC filing year."
    )

    query: str = Field(
        description="The user's actual financial question or request."
    )

In [15]:
import json


def analyze_query(user_question: str) -> SearchQuery:

    prompt = f"""
You are the query analyzer for FinSight AI.

Extract the following information from the user's request:

1. intent
2. company
3. filing_year
4. query

Allowed intents:
- question_answering
- filing_retrieval

Allowed companies:
- AAPL
- MSFT
- AMZN
- NVDA
- GOOGL

Return ONLY valid JSON.
Do not include markdown or explanations.

User request:
{user_question}

JSON format:
{{
    "intent": "question_answering",
    "company": "AMZN",
    "filing_year": "2025",
    "query": "What did Amazon say about AWS in 2025?"
}}
"""

    response = llm.invoke(prompt)

    data = json.loads(response.content)

    return SearchQuery(**data)

NameError: name 'SearchQuery' is not defined

In [20]:
from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from Pinecone."""

    def __init__(self, index, embedding_manager):
        self.index = index
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        company: str,
        filing_year: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ):
        print(f"Retrieving documents for: '{query}'")
        print(f"Company: {company}")
        print(f"Filing year: {filing_year}")
        print(f"Top K: {top_k}")

        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        try:
            results = self.index.query(
                vector=query_embedding.tolist(),
                top_k=top_k,
                include_metadata=True,
                filter={
                    "company": {"$eq": company},
                    "filing_year": {"$eq": filing_year}
                }
            )

            retrieved_docs = []

            for rank, match in enumerate(results.matches, start=1):

                if match.score >= score_threshold:
                    retrieved_docs.append({
                        "id": match.id,
                        "content": match.metadata.get("text"),
                        "metadata": match.metadata,
                        "similarity_score": match.score,
                        "rank": rank
                    })

            print(f"Retrieved {len(retrieved_docs)} documents")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [21]:
rag_retriever = RAGRetriever(
    index=index,
    embedding_manager=embeddingmanager
)

In [ ]:
def rag_simple(query, rag_retriever, llm, top_k=10):
    
    # Step 1: Understand the user's question
    search_query = analyze_query(query)

    print("\n--- Query Analysis ---")
    print("Intent:", search_query.intent)
    print("Company:", search_query.company)
    print("Filing Year:", search_query.filing_year)
    print("Query:", search_query.query)

    # Step 2: Retrieve relevant chunks
    results = rag_retriever.retrieve(
        query=search_query.query,
        company=search_query.company,
        filing_year=search_query.filing_year,
        top_k=top_k
    )

    context = "\n\n".join(
        f"[Source {i+1}]\n{doc['content']}"
        for i, doc in enumerate(results)
    ) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Step 3: Generate the answer
    prompt = f"""
You are FinSight AI, a financial research assistant.

Answer the user's question using ONLY the information provided
in the SEC filing context.

If the context does not contain enough information, say:

"I don't have enough information in the provided SEC filings."

Do not use outside knowledge.
Do not make up financial information.

Company: {search_query.company}

Filing Year: {search_query.filing_year}

Question:
{search_query.query}

SEC Filing Context:
{context}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content